### Sistema Recomendación Híbrido

El objetivo de este proyecto es construir un sistema de recomendación basado en las compras realizadas por los clientes de una superficie comercial. El enfoque parte de una restricción clave: al tratarse de compra física, no es posible recomendar en tiempo real durante la compra, ya que no conocemos la cesta del cliente hasta que pasa por caja. Esto diferencia el problema del de una tienda online, donde sí se puede recomendar según la cesta se va componiendo.
Por ello, el sistema se orienta a la recomendación entre visitas: a partir del historial de compra de cada cliente, generar recomendaciones personalizadas que incentiven su siguiente visita (mediante email, app o cupones en el ticket), en línea con los programas de fidelización habituales en el sector.
Para ello se construyen tres modelos:

- ALS, que aprende los hábitos de cada cliente a partir de su historial para predecir y recomendar su próxima compra. Es el núcleo de la personalización.
- Apriori, que analiza la composición de las cestas para extraer patrones de co-compra. Más que recomendar al cliente directamente, alimenta decisiones de negocio: colocación de productos en tienda, promociones cruzadas y cupones post-compra.
- Popularidad, que recomienda los productos más vendidos como red de seguridad para clientes nuevos o sin historial suficiente.

### Librerías

In [1]:
import pandas as pd

In [6]:
df = pd.read_pickle('data/transacciones_limpio.pkl')

df.head(5)

,line_id,ticket_number,user_card_id,payment_method,shop_name,shop_address,product_code,product_name,seccion,units,price_per_unit,price_total,created_at
0,9550698,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1041,HELADO VAINILLA 1L,Congelados,1,3.41,3.41,2025-01-01 08:20:00
1,9550697,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1008,AZUCAR BLANCO 1KG,Alimentacion seca,1,1.06,1.06,2025-01-01 08:20:00
2,9550707,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1055,BOLSAS BASURA 30L,Drogueria,1,1.98,1.98,2025-01-01 08:20:00
3,9550706,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1022,CARNE PICADA MIXTA 500G,Frescos,1,3.95,3.95,2025-01-01 08:20:00
4,9550693,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1068,ARENA GATO 5L,Mascotas,2,3.94,7.88,2025-01-01 08:20:00


Comprobamos longitud datos y fecha mínima y máxima para realizar división de datos

In [9]:
print(f'Filas: {len(df):,}')
print(df.created_at.min())
print(df.created_at.max())

Filas: 50,000
2025-01-01 08:20:00
2025-06-30 21:21:00


Tras comprobar fechas procedemos a separar los datos y crear un conjunto de entrenamiento y testeo. Esta separación se realiza de forma manual.

In [ ]:
# Fecha corte. Últimos 30 días para test
fecha_corte = df.created_at.max() - pd.Timedelta(days=30)
print(f'Fecha corte: {fecha_corte}')

# creamos conjuntos
train = df[df.created_at <= fecha_corte].copy()
test = df[df.created_at > fecha_corte].copy()

print(f'Longitud Train: {len(train)}; ({len(train)/len(df)*100:.0f}%)') 
print(f'Longitud Test: {len(test)}; ({len(test)/len(df)*100:.0f}%)')
print('-'*50)
print(f'Tickets train: {train.ticket_number.nunique():,}') # comprobación de tickets por conjunto
print(f'Tickets test: {test.ticket_number.nunique():,}')


Fecha corte: 2025-05-31 21:21:00
Longitud Train: 41649; (83%)
Longitud Test: 8351; (17%)
--------------------------------------------------
Tickets train: 4,340
Tickets test: 856


Ahora vamos a comprobar el número de clientes evaluables. Este paso es importante porque determina la frontera a partir de la cual entra en juego cada modelo. Para ALS, un sistema personalizado, se necesita un registro histórico del cliente por tanto no funcionará con aquellos clientes nuevos, lo que se conoce como cold start. Estos clientes de cold start serán asignados a sistemas como Apriori o popularidad para realizar la recomendación.

In [ ]:
# almacenamos los clientes en conjuntos y eliminamos duplicados
clientes_train = set(train.user_card_id.unique())
clientes_test = set(test.user_card_id.unique())

evaluables = clientes_test & clientes_train # intersección que guarda solo clientes que se encuentran en ambos conjuntos. Aparecen tanto en train como test
cold_start = clientes_test - clientes_train  # clientes que compraron por primera vez dentro del rango de test.

print(f'Clientes en test: {len(clientes_test)}')
print(f'Evaluables: {len(evaluables)}') # clientes sobre los que se puede probar ALS
print(f'Cold start puro (test): {len(cold_start)}')

Clientes en test: 583
Evaluables: 537
Cold start puro (test): 46


Con estos cálculos ya tenemos una idea sobre la cantidad de clientes que vamos a manejar en las evaluaciones de modelos.
Contamos con 583 clientes dentro del conjunto de testeo de los cuales, 46 de ellos han realizado su primera compra en el periodo que contempla el conjunto de test. Los restantes 537 clientes se encuentran en ambos conjuntos y nos ayudarán a medir si ALS acierta. 

### Modelo Basado en Popularidad

El primer modelo que se construye es un sistema NO personalizado. Este modelo solo se va a fijar en cuantas veces aparece cada producto en cada ticket. Se trata de un modelo totalmente sencillo que no necesita conocer ningún cliente, es ideal para cuando estamos empezando o el cliente es nuevo, se usa como referencia base.

In [ ]:
# Popularidad de producto según tickets en conjunto train
popularidad = (
    train.groupby('product_name')['ticket_number']
    .nunique()
    .sort_values(ascending=False)
)

print('Top 10 productos más populares (por nº de tickets):')
print(popularidad.head(10))

Top 10 productos más populares (por nº de tickets):
product_name
CHAMPU ANTICASPA 400ML      3393
LECHUGA ICEBERG UD          2843
YOGUR NATURAL PACK 4        2231
HELADO VAINILLA 1L          1855
ARROZ REDONDO 1KG           1643
TOMATE RAMA KG              1610
ENSALADA CESAR PREPARADA    1336
CERVEZA PACK 6              1264
SAL FINA 1KG                1117
ARENA GATO 5L               1073
Name: ticket_number, dtype: int64


In [ ]:
def recomendar_popularidad(n=10):
    """
    Función para recomendar productos en base a la popularidad. Veces que aparece en tickets
    Parámetros:
    -n: número de productos seleccionados
    """
    return popularidad.head(n).index.tolist()

print(recomendar_popularidad(10))

['CHAMPU ANTICASPA 400ML', 'LECHUGA ICEBERG UD', 'YOGUR NATURAL PACK 4', 'HELADO VAINILLA 1L', 'ARROZ REDONDO 1KG', 'TOMATE RAMA KG', 'ENSALADA CESAR PREPARADA', 'CERVEZA PACK 6', 'SAL FINA 1KG', 'ARENA GATO 5L']


Vamos a evaluar el comportamiento de este sistema base

In [34]:
recomendados = set(popularidad.head(10).index) # creamos un conjunto con los 10 productos más populares

# Comprobamos cliente de forma individual
cliente = list(evaluables)[0] # creamos una lista con los clientes y seleccionamos el primer cliente
comprados = set(test[test['user_card_id'] == cliente]['product_name']) # conjunto de productos comprados por el cliente dentro de test

print(f'Cliente: {cliente}')
print(f'Productos comprados en conjunto test: {len(comprados)}')
print(comprados)

Cliente: 100352
Productos comprados en conjunto test: 23
{'HELADO VAINILLA 1L', 'ARROZ REDONDO 1KG', 'PAPILLA CEREALES 600G', 'LECHE ENTERA BRIK 1L', 'HUEVOS DOCENA M', 'LECHE INFANTIL CONTINUACION', 'TOMATE RAMA KG', 'MACARRONES 500G', 'CAFE MOLIDO 250G', 'SAL FINA 1KG', 'PIZZA CUATRO QUESOS', 'ARENA GATO 5L', 'TOMATE FRITO 400G', 'MANZANA GOLDEN KG', 'REFRESCO COLA 2L', 'YOGUR NATURAL PACK 4', 'CHAMPU ANTICASPA 400ML', 'PECHUGA POLLO BANDEJA', 'AZUCAR BLANCO 1KG', 'LECHUGA ICEBERG UD', 'PATATA KG', 'MERLUZA FILETE KG', 'ENSALADA CESAR PREPARADA'}


Guardamos en una variable los productos comprados por un cliente concreto durante el periodo de testeo. Ahora vamos a comparar este cliente con el top de productos más populares y ver la tasa de acierto.

In [36]:
aciertos = recomendados & comprados # creamos una intersección entre productos recomendados y comprados del cliente seleccionado

print(f'Nº de aciertos: {len(aciertos)}')
print(f'Aciertos: {aciertos}')

Nº de aciertos: 9
Aciertos: {'HELADO VAINILLA 1L', 'ARENA GATO 5L', 'ARROZ REDONDO 1KG', 'YOGUR NATURAL PACK 4', 'CHAMPU ANTICASPA 400ML', 'TOMATE RAMA KG', 'LECHUGA ICEBERG UD', 'ENSALADA CESAR PREPARADA', 'SAL FINA 1KG'}


De los 23 productos comprados por el cliente en el conjunto de test, 9 de ellos se encuentran dentro de los 10 más populares

In [ ]:
precision = len(aciertos) / 10 # cuantos valores acertamos
recall = len(aciertos) / len(comprados) # de los productos comprados, cuantos cubrimos

print(f'Precision@10 para el cliente: {precision:.2f}')
print(f'Recall@10 para el cliente: {recall:.2f}')

Precision@10 para el cliente: 0.90
Recall@10 para el cliente: 0.39
